In [5]:
import json
import os
import glob

# 1. 定义输入文件夹路径和输出文件路径
input_dir = r'C:\Users\18056\OneDrive\Desktop\研一\nlp\汽车模型\TARA\数据\DAD\test'
output_file = r'C:\Users\18056\OneDrive\Desktop\研一\nlp\汽车模型\TARA\数据\DAD\output\本地升级.json'

# 初始化一个空列表，用于存储所有文件的汇总结果
all_results = []

# 【新增】：定义 type 的映射字典
TYPE_MAPPING = {
    "tm.Flow": "信号",
    "tm.Process": "部件",
    "tm.Store": "数据",
    "tm.Actor": "接口"
}

# 2. 获取文件夹下所有的 .json 文件
json_files = glob.glob(os.path.join(input_dir, '*.json'))

print(f"共找到 {len(json_files)} 个 JSON 文件，开始处理...")

# 3. 遍历处理每个文件
for file_path in json_files:
    if os.path.abspath(file_path) == os.path.abspath(output_file):
        continue

    with open(file_path, 'r', encoding='utf-8') as f:
        try:
            data = json.load(f)
        except json.JSONDecodeError:
            print(f"⚠️ 解析失败，跳过非标准 JSON 文件: {os.path.basename(file_path)}")
            continue

    current_result = {'function': '', 'assets': []}

    if 'detail' in data and 'diagrams' in data['detail']:
        for diagram in data['detail']['diagrams']:
            
            if not current_result['function'] and 'title' in diagram:
                current_result['function'] = diagram['title']
                
            if 'diagramJson' in diagram and 'cells' in diagram['diagramJson']:
                for cell in diagram['diagramJson']['cells']:
                    
                    if cell.get('outOfScope') is True:
                        continue
                    
                    # 先获取原始的 type 值
                    raw_cell_type = cell.get('type', '')
                    
                    # 排除 tm.Boundary（使用原始值进行判断）
                    if raw_cell_type == 'tm.Boundary':
                        continue
                    
                    # 提取 finetermval 的值
                    finetermval_value = ""
                    for key, value in cell.items():
                        if key.startswith('propertyList') and isinstance(value, dict):
                            if 'finetermval' in value:
                                finetermval_value = value['finetermval']
                                break
                    
                    if not finetermval_value:
                        continue
                    
                    # 根据映射字典转换 type，如果没匹配到则保留原值
                    mapped_cell_type = TYPE_MAPPING.get(raw_cell_type, raw_cell_type)
                    
                    # 组装新 cell，使用转换后的 mapped_cell_type
                    current_result['assets'].append({
                        'asset_type': mapped_cell_type,
                        'asset_name': finetermval_value
                    })

    if not current_result['function']:
        current_result['function'] = os.path.splitext(os.path.basename(file_path))[0]

    if current_result['assets']:
        all_results.append(current_result)
        print(f"✅ 成功处理: {current_result['function']} (提取了 {len(current_result['assets'])} 个节点)")
    else:
        print(f"⏭️ 跳过文件: {os.path.basename(file_path)} (未提取到有效节点)")

# 4. 将汇总结果写入单个 JSON 文件
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(all_results, f, ensure_ascii=False, indent=4)

print('=' * 40)
print(f"🎉 全部处理完成！")
print(f"共提取了 {len(all_results)} 个有效拓扑图模型。")
print(f"汇总文件已保存至: {output_file}")


共找到 1 个 JSON 文件，开始处理...
✅ 成功处理: 本地升级 (提取了 13 个节点)
🎉 全部处理完成！
共提取了 1 个有效拓扑图模型。
汇总文件已保存至: C:\Users\18056\OneDrive\Desktop\研一\nlp\汽车模型\TARA\数据\DAD\output\本地升级.json


In [ ]:
source

In [9]:
import json
import os
import glob

# 1. 定义输入文件夹路径和输出文件路径
input_dir = r'C:\Users\18056\OneDrive\Desktop\研一\nlp\汽车模型\TARA\数据\DAD\input'
output_file = r'C:\Users\18056\OneDrive\Desktop\研一\nlp\汽车模型\TARA\数据\DAD\output\DAD_Summary.json'

# 初始化一个空列表，用于存储所有文件的汇总结果
all_results = []

# 【新增】：定义 type 的映射字典
TYPE_MAPPING = {
    "tm.Flow": "信号",
    "tm.Process": "部件",
    "tm.Store": "数据",
    "tm.Actor": "接口"
}

# 2. 获取文件夹下所有的 .json 文件
json_files = glob.glob(os.path.join(input_dir, '*.json'))

print(f"共找到 {len(json_files)} 个 JSON 文件，开始处理...")

# 3. 遍历处理每个文件
for file_path in json_files:
    if os.path.abspath(file_path) == os.path.abspath(output_file):
        continue

    with open(file_path, 'r', encoding='utf-8') as f:
        try:
            data = json.load(f)
        except json.JSONDecodeError:
            print(f"⚠️ 解析失败，跳过非标准 JSON 文件: {os.path.basename(file_path)}")
            continue

    current_result = {'function': '', 'assets': []}

    if 'detail' in data and 'diagrams' in data['detail']:
        for diagram in data['detail']['diagrams']:
            
            if not current_result['function'] and 'title' in diagram:
                current_result['function'] = diagram['title']
                
            if 'diagramJson' in diagram and 'cells' in diagram['diagramJson']:
                cells = diagram['diagramJson']['cells']
                
                # 建立 id 到 cell 的映射，用于查找 source 和 target
                id_to_cell = {}
                for cell in cells:
                    if 'id' in cell:
                        id_to_cell[cell['id']] = cell
                
                for cell in cells:
                    
                    if cell.get('outOfScope') is True:
                        continue
                    
                    # 先获取原始的 type 值
                    raw_cell_type = cell.get('type', '')
                    
                    # 排除 tm.Boundary（使用原始值进行判断）
                    if raw_cell_type == 'tm.Boundary':
                        continue
                    
                    # 提取 assetDetails 和 finetermval 的值
                    asset_details = ""
                    finetermval_value = ""
                    for key, value in cell.items():
                        if key.startswith('propertyList') and isinstance(value, dict):
                            if 'assetDetails' in value:
                                asset_details = value['assetDetails']
                            if 'finetermval' in value:
                                finetermval_value = value['finetermval']
                            break
                    
                    if not finetermval_value:
                        continue
                    
                    # 根据映射字典转换 type，如果没匹配到则保留原值
                    mapped_cell_type = TYPE_MAPPING.get(raw_cell_type, raw_cell_type)
                    
                    # 根据类型确定 target 和 source
                    target_value = ""
                    source_value = ""
                    
                    if mapped_cell_type == "部件":
                        # "asset_type"为"部件"的"target"等于其"asset_name"
                        target_value = finetermval_value
                        
                    elif mapped_cell_type == "数据":
                        # "asset_type"为"数据"的"target"等于以其 id 为 source 的"type": "tm.Flow"的 target.id 对应的"finetermval"
                        cell_id = cell.get('id', '')
                        for flow_cell in cells:
                            if (flow_cell.get('type') == 'tm.Flow' and 
                                flow_cell.get('source', {}).get('id') == cell_id):
                                target_id = flow_cell.get('target', {}).get('id', '')
                                if target_id in id_to_cell:
                                    target_cell = id_to_cell[target_id]
                                    for key, value in target_cell.items():
                                        if key.startswith('propertyList') and isinstance(value, dict):
                                            if 'finetermval' in value:
                                                target_value = value['finetermval']
                                                break
                                    if target_value:
                                        break
                    
                    elif mapped_cell_type == "信号":
                        # "asset_type"为"信号"的"target"等于其 target.id 对应的"finetermval"
                        # "source"等于其 source.id 对应的"finetermval"
                        target_id = cell.get('target', {}).get('id', '')
                        source_id = cell.get('source', {}).get('id', '')
                        
                        if target_id in id_to_cell:
                            target_cell = id_to_cell[target_id]
                            for key, value in target_cell.items():
                                if key.startswith('propertyList') and isinstance(value, dict):
                                    if 'finetermval' in value:
                                        target_value = value['finetermval']
                                        break
                        
                        if source_id in id_to_cell:
                            source_cell = id_to_cell[source_id]
                            for key, value in source_cell.items():
                                if key.startswith('propertyList') and isinstance(value, dict):
                                    if 'finetermval' in value:
                                        source_value = value['finetermval']
                                        break
                    
                    # 组装新 cell，使用转换后的 mapped_cell_type
                    asset_entry = {
                        'asset_type': mapped_cell_type,
                        'asset_name': finetermval_value,
                        'assetDetails': asset_details,
                    }
                    
                    if target_value:
                        asset_entry['target'] = target_value
                    
                    if source_value:
                        asset_entry['source'] = source_value
                    
                    current_result['assets'].append(asset_entry)

    if not current_result['function']:
        current_result['function'] = os.path.splitext(os.path.basename(file_path))[0]

    if current_result['assets']:
        all_results.append(current_result)
        print(f"[OK] 成功处理：{current_result['function']} (提取了 {len(current_result['assets'])} 个节点)")
    else:
        print(f"[SKIP] 跳过文件：{os.path.basename(file_path)} (未提取到有效节点)")

# 4. 将汇总结果写入单个 JSON 文件
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(all_results, f, ensure_ascii=False, indent=4)

print('=' * 40)
print(f"[SUCCESS] 全部处理完成！")
print(f"共提取了 {len(all_results)} 个有效拓扑图模型。")
print(f"汇总文件已保存至: {output_file}")


共找到 3 个 JSON 文件，开始处理...
[OK] 成功处理：交通标识及信号灯识别 (提取了 3 个节点)
[OK] 成功处理：智能泊车辅助 (提取了 8 个节点)
[OK] 成功处理：碰撞防护 (提取了 7 个节点)
[SUCCESS] 全部处理完成！
共提取了 3 个有效拓扑图模型。
汇总文件已保存至: C:\Users\18056\OneDrive\Desktop\研一\nlp\汽车模型\TARA\数据\DAD\output\DAD_Summary.json
